In [ ]:
!pip install nle minigrid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 42.5 MB/s  0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.4.0
    Uninstalling gymnasium-1.4.0:
      Successfully uninstalled gymnasium-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [minigrid]


In [ ]:
import gymnasium as gym
import torch
import numpy as np
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
import minigrid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import random

from torch.distributions import Categorical
from tqdm.auto import trange

In [ ]:
# Среда и воспроизводимость.
SEED = 42
ENV_ID = "MiniGrid-DoorKey-8x8-v0"
GRID_SIZE = 8
MAX_EPISODE_STEPS = 320
NUM_ACTIONS = 6             # 0: left, 1: right, 2: forward.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))  # Маленькие батчи быстрее на CPU.
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"  # На CPU сохраняем float32.


# Небольшой вариант эксперимента, рассчитанный на Colab.
NUM_ENVS = 8
ROLLOUT_LENGTH = 128
NUM_UPDATES = 500           # Для быстрого smoke-test поставьте 2–5.
MINIBATCH_SIZE = 128
TEACHER_EPOCHS = 1
PPO_EPOCHS = 2
LEARNING_RATE = 2e-4
HIDDEN = 128                # В статье: 1024 (более дорогое обучение).
EMBED_DIM = 64             # Размер CRL-представлений в статье.


# Коэффициенты дисконта и PPO.
GAMMA = 0.995              # Дисконт будущих целей и PPO (как в статье).
GAMMA_TERM = 0.8           # Штраф за смерть (пример абляции статьи).
K_MAX = 256               # Максимальная глубина будущего в CRL.
GAE_LAMBDA = 0.95
PPO_CLIP = 0.2
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
BC_START = 0.1             # В статье β=0.1 с последующим уменьшением до 0.
MAX_GRAD_NORM = 1.0

print("Device:", DEVICE)

In [ ]:
def observe_converter(observation, env):
    """Превращает наблюдение MiniGrid в два компактных входа для сетей.

    Задача:
        Изолировать от Teacher/Student формат API конкретной среды.
        Сети не знают о gym, объектах Grid и правилах MiniGrid.

    Возвращает:
        image: uint8 [7, 7, 3] — частичная эгоцентрическая карта;
        extra: float32 [6] — x, y в [0,1] и one-hot направление (4 числа).
    """
    image = observation["image"].astype(np.uint8)

    # Направление MiniGrid: 0=right, 1=down, 2=left, 3=up.
    direction = int(observation["direction"])
    direction_onehot = np.eye(4, dtype=np.float32)[direction]

    # Делаем координаты независимыми от размера карты.
    x, y = env.unwrapped.agent_pos
    width = env.unwrapped.width
    height = env.unwrapped.height
    coords = np.array([x / (width - 1), y / (height - 1)], dtype=np.float32)
    carry = np.array([env.unwrapped.carrying is not None and env.unwrapped.carrying.type == "key"], dtype=np.float32)

    extra = np.concatenate([coords, direction_onehot, carry])
    return image, extra

Цели для MiniGrid-DoorKey-8x8-v0

In [ ]:
def object_position(env):
    base = env.unwrapped

    positions = {
        "key": None,
        "door": None,
        "goal": None,
    }
    for x in range(base.width):
        for y in range(base.height):
            obj = base.grid.get(x, y)

            if obj is not None and obj.type in positions:
                positions[obj.type] = (x, y)
    return positions

In [ ]:
GOAL_NAMES = [
    "upper_half_start", "lower_half_start",
    "near_key", "has_key",
    "near_door",
    "near_door_wkey", "open_door", "come_through_door",
    "second_room", "upper_half_second", "lower_half_second",
    "near_exit", "at_exit"
]
NUM_GOALS = len(GOAL_NAMES)

In [ ]:
def goal_predicates(env, extra, positions):
    # Восстанавливаем целочисленные координаты из нормализованных.
    x = int(round(float(extra[0]) * (GRID_SIZE - 1)))
    y = int(round(float(extra[1]) * (GRID_SIZE - 1)))
    direction = int(np.argmax(extra[2:]))

    wall_x = positions["door"][0]
    door_y = positions["door"][1]
    key_x = positions["key"][0]
    key_y = positions["key"][1]
    middle_y = (GRID_SIZE - 1) // 2

    key_obj = env.unwrapped.grid.get(key_x, key_y)
    key_on_floor = key_obj is not None and key_obj.type == "key"

    near_key = (
        key_on_floor
        and abs(x - key_x) + abs(y - key_y) <= 1
    )

    has_key = (
        env.unwrapped.carrying is not None
        and env.unwrapped.carrying.type == "key"
    )

    return np.array([
        x <= wall_x and y >= middle_y,
        x <= wall_x and y <= middle_y,
        near_key,
        has_key,
        abs(x-wall_x) + abs(y-door_y) <= 1,
        abs(x-wall_x) + abs(y-door_y) <= 1 and env.unwrapped.carrying is not None,
        env.unwrapped.grid.get(wall_x, door_y).is_open,
        x == wall_x and y == door_y,
        x > wall_x,
        x > wall_x and y >= middle_y,
        x > wall_x and y <= middle_y,
        abs(x-(GRID_SIZE - 2)) + abs(y-(GRID_SIZE - 2)) <= 1,
        x == GRID_SIZE - 2 and y == GRID_SIZE - 2
    ], dtype=np.float32)

In [ ]:
"""
example_env = gym.make(ENV_ID, max_steps=MAX_EPISODE_STEPS)
example_observation, _ = example_env.reset(seed=SEED)
example_image, example_extra = observe_converter(example_observation, example_env)
print("Image:", example_image.shape, "Extra:", example_extra.shape)
print("Достигнутые цели на старте:", [name for name, hit in zip(GOAL_NAMES, goal_predicates(example_extra)) if hit])
example_env.close()
"""

'\nexample_env = gym.make(ENV_ID, max_steps=MAX_EPISODE_STEPS)\nexample_observation, _ = example_env.reset(seed=SEED)\nexample_image, example_extra = observe_converter(example_observation, example_env)\nprint("Image:", example_image.shape, "Extra:", example_extra.shape)\nprint("Достигнутые цели на старте:", [name for name, hit in zip(GOAL_NAMES, goal_predicates(example_extra)) if hit])\nexample_env.close()\n'

In [ ]:
class ObservationEncoder(nn.Module):
    def __init__(self, view_size=7, hidden_dim=HIDDEN):
        super().__init__()

        # Разные таблицы для типа объекта, цвета и состояния двери.
        self.object_embedding = nn.Embedding(16, 8)
        self.color_embedding = nn.Embedding(16, 4)
        self.state_embedding = nn.Embedding(8, 4)

        # Одна 3x3 свёртка на 32 канала — близко к backbone статьи.
        self.conv = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        # +6 — координаты x/y и one-hot направление из observe().
        self.mlp = nn.Sequential(
            nn.Linear(32 * view_size * view_size + 7, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, image, extra):
        """image [B,7,7,3], extra [B,6] -> h(s) [B,HIDDEN]."""
        # Представляем каждую категорию обучаемым вектором.
        objects = self.object_embedding(image[..., 0].long())
        colors = self.color_embedding(image[..., 1].long())
        states = self.state_embedding(image[..., 2].long())
        tiles = torch.cat([objects, colors, states], dim=-1)

        # PyTorch Conv2d ожидает формат [B, channels, height, width].
        tiles = tiles.permute(0, 3, 1, 2)
        spatial = F.relu(self.conv(tiles)).flatten(start_dim=1)

        # Совмещаем карту и статистику агента только после свёртки.
        return self.mlp(torch.cat([spatial, extra.float()], dim=-1))

In [ ]:
@torch.no_grad()
def rao_blackwell_targets(achieved, done, fatal, gamma=GAMMA,
                             gamma_term=GAMMA_TERM, k_max=K_MAX):
    """Строит распределения целей для ВСЕХ переходов rollout.

    Задача:
        Убрать случайное сэмплирование будущего offset и случайную
        метку из множества достигнутых целей; усреднить точно по всем
        наблюдаемым будущим состояниям (Rao–Blackwell, формулы 6–8).

    Вход:
        achieved: [T,N,G], индикаторы ВСЕХ целей в следующем состоянии;
        done:     [T,N], конец эпизода после соответствующего действия;
        fatal:    [T,N], необратимая гибель (подмножество done).

    Выход:
        targets: [T,N,G+1], распределение (G целей + ∅), сумма = 1.

    Важно:
        Эпизоды внутри rollout могут обрываться в разное время.
        Поэтому для каждой среды нормируем веса на ЕЁ число доступных
        будущих шагов, не перемешивая эпизоды между собой.
    """
    T, N, G = achieved.shape
    targets = torch.zeros(T, N, G + 1, device=achieved.device)

    # В одном будущем состоянии разделяем единицу между всеми true-целями.
    satisfied_count = achieved.sum(dim=-1, keepdim=True).clamp_min(1.0)
    per_state_labels = achieved / satisfied_count

    for t in range(T):
        weighted_goals = torch.zeros(N, G, device=achieved.device)
        total_weights = torch.zeros(N, device=achieved.device)
        death_distance = torch.zeros(N, device=achieved.device)
        alive = torch.ones(N, device=achieved.device, dtype=torch.bool)

        for k in range(1, min(k_max, T - t) + 1):
            # Геометрическое распределение шагов в будущее.
            weight = (1.0 - gamma) * (gamma ** (k - 1))
            weighted_goals += weight * alive[:, None] * per_state_labels[t + k - 1]
            total_weights += weight * alive

            # Сохраняем расстояние до ПЕРВОЙ наблюдаемой гибели.
            just_died = alive & fatal[t + k - 1] & (death_distance == 0)
            death_distance[just_died] = k

            # Последнее состояние эпизода уже включено в weighted_goals.
            # Но состояние после reset в будущее этого anchor не включаем.
            alive = alive & (~done[t + k - 1])
            if not alive.any():
                break

        # Нормализация на реально наблюдённые будущие шаги (формула 6).
        goal_distribution = weighted_goals / total_weights.clamp_min(1e-8)[:, None]

        # Штраф γ_term^Δ применяется только к реальной гибели.
        survived_mass = torch.where(
            death_distance > 0,
            1.0 - gamma_term ** death_distance,
            1.0,
        )
        goal_distribution *= survived_mass[:, None]

        # Остаток массы всегда получает поглощающий ∅.
        targets[t, :, :G] = goal_distribution
        targets[t, :, G] = 1.0 - goal_distribution.sum(dim=-1)

    return targets

In [ ]:
class Teacher(nn.Module):
    # Обучаемый CRL-критик, который выбирает действия для любой цели.
    def __init__(self, num_actions=NUM_ACTIONS, num_goals=NUM_GOALS, hidden_dim=HIDDEN, embed_dim=EMBED_DIM):
        super().__init__()
        self.num_actions = num_actions
        self.num_goals = num_goals

        self.encoder = ObservationEncoder()

        # Каждому дискретному действию соответствует свой вектор φ(s,a).
        # Одна линейная проекция сразу выдаёт num_actions векторов.
        self.state_action = nn.Linear(hidden_dim, num_actions * embed_dim)

        # ψ(g) — отдельное обучаемое представление каждой семантической цели.
        self.goal_embedding = nn.Embedding(num_goals, embed_dim)

    def forward(self, image, extra):
        """Оценки f(s,a,g) всех действий и всех целей: [B, A, G]."""
        state_embed = self.encoder(image, extra)
        state_actions = self.state_action(state_embed).reshape(-1, self.num_actions, EMBED_DIM)
        goals = self.goal_embedding.weight

        # Скалярное произведение φ(s,a)·ψ(g), без нормализации и softmax.
        return torch.einsum("bad,gd->bag", state_actions, goals)

    def loss(self, image, extra, actions, targets):
        """CRL cross-entropy по распределению будущих целей.

        actions: [B] — действие, реально совершённое в rollout.
        targets: [B,G+1] — мягкие метки от rao_blackwell_targets().

        Учим f(s,a,g) для ФАКТИЧЕСКОГО a; прогнозы для других a
        обучаются, когда они встречаются в новых траекториях.
        Последний столбец targets — вероятность поглощающего ∅.
        """
        all_logits = self.forward(image, extra)
        batch_index = torch.arange(actions.shape[0], device=actions.device)
        chosen_logits = all_logits[batch_index, actions]  # [B,G]

        # У absorbing class ∅ фиксированный логит 0 (формула 7).
        absorbing_logits = torch.zeros_like(chosen_logits[:, :1])
        logits = torch.cat([chosen_logits, absorbing_logits], dim=-1)

        # Cross-entropy с распределением, а не с одной целевой меткой.
        log_probs = F.log_softmax(logits.float(), dim=-1)  # CE считаем в float32 при AMP.
        return -(targets * log_probs).sum(dim=-1).mean()

    @torch.no_grad()
    def greedy_action(self, image, extra, goal_ids):
        """Возвращает a*=argmax_a f(s,a,g) для выбранной цели [B].

        Это действие выступает целевой меткой для behavioral cloning
        студента, но сами веса учителя не меняются из BC-loss.
        """
        all_scores = self.forward(image, extra)  # [B,A,G]
        chosen_scores = all_scores.gather(
            dim=2,
            index=goal_ids[:, None, None].expand(-1, self.num_actions, 1),
        ).squeeze(-1)                           # [B,A]
        return chosen_scores.argmax(dim=-1)

In [ ]:
class Student(nn.Module):
    """Goal-conditioned PPO actor–critic, обучающийся с помощью Teacher.

    Задача:
        Сформировать собственную стратегию π(a|s,g) и оценку V(s,g),
        пользуясь одновременно PPO-наградой и подсказками от учителя.

    Отличие от Teacher:
        Teacher оценивает ВСЕ цели и ВСЕ действия, а Student принимает
        ОДНУ командованную цель и выдаёт распределение по действиям.
        Ученик не копирует веса Teacher, а лишь его argmax-действие.
    """
    def __init__(self, num_actions=NUM_ACTIONS, num_goals=NUM_GOALS):
        super().__init__()
        self.encoder = ObservationEncoder()
        self.goal_embedding = nn.Embedding(num_goals, 32)

        # Объединяем представление текущего состояния и команды g.
        self.policy_body = nn.Sequential(
            nn.Linear(HIDDEN + 32, HIDDEN),
            nn.ReLU(),
        )

        # Actor: логиты действий. Critic: value для PPO/GAE.
        self.actor = nn.Linear(HIDDEN, num_actions)
        self.critic = nn.Linear(HIDDEN, 1)

    def forward(self, image, extra, goal_ids):
        """Возвращает (action_logits [B,A], state_values [B])."""
        state = self.encoder(image, extra)
        goal = self.goal_embedding(goal_ids.long())
        features = self.policy_body(torch.cat([state, goal], dim=-1))

        return self.actor(features), self.critic(features).squeeze(-1)

In [ ]:
"""
# Тестируем разные комбинации целей и фатальный переход.
test_achieved = torch.tensor([
    [[1., 1., 0.], [0., 1., 0.]],
    [[0., 0., 1.], [0., 0., 0.]],
])  # [T=2,N=2,G=3]
test_done = torch.tensor([[False, False], [True, True]])
test_fatal = torch.tensor([[False, False], [False, True]])

labels = rao_blackwell_targets(test_achieved, test_done, test_fatal,
                               gamma=0.9, gamma_term=0.8, k_max=2)
assert labels.shape == (2, 2, 4)
assert torch.allclose(labels.sum(-1), torch.ones(2, 2), atol=1e-6)
assert torch.all(labels >= -1e-6)
assert labels[1, 1, -1] == 1.0  # Фатальное действие без достигнутых целей.

# При гибели следующая цель получает (1-γ_term), при успехе — всю массу.
test_positive = torch.ones(1, 2, 1)
test_end = torch.ones(1, 2, dtype=torch.bool)
test_is_fatal = torch.tensor([[True, False]])
test_terminal = rao_blackwell_targets(
    test_positive, test_end, test_is_fatal, gamma=0.9, gamma_term=0.8
)
assert torch.allclose(test_terminal[0, :, 0], torch.tensor([0.2, 1.0]), atol=1e-6)
assert torch.allclose(test_terminal[0, :, 1], torch.tensor([0.8, 0.0]), atol=1e-6)

# Проверяем также обратное распространение через оба типа модели.
test_image = torch.zeros(4, 7, 7, 3, dtype=torch.long, device=DEVICE)
test_extra = torch.zeros(4, 6, device=DEVICE)
test_goal_ids = torch.tensor([0, 1, 2, 3], device=DEVICE)
test_actions = torch.tensor([0, 1, 2, 0], device=DEVICE)

test_teacher = Teacher().to(DEVICE)
test_student = Student().to(DEVICE)
test_y = torch.zeros(4, NUM_GOALS + 1, device=DEVICE)
test_y[:, -1] = 1.0

test_teacher.loss(test_image, test_extra, test_actions, test_y).backward()
test_logits, test_values = test_student(test_image, test_extra, test_goal_ids)
assert test_logits.shape == (4, NUM_ACTIONS) and test_values.shape == (4,)
(test_logits.mean() + test_values.mean()).backward()
print("Тесты passed: формы, градиенты, mass conservation, fatal transitions")
"""

'\n# Тестируем разные комбинации целей и фатальный переход.\ntest_achieved = torch.tensor([\n    [[1., 1., 0.], [0., 1., 0.]],\n    [[0., 0., 1.], [0., 0., 0.]],\n])  # [T=2,N=2,G=3]\ntest_done = torch.tensor([[False, False], [True, True]])\ntest_fatal = torch.tensor([[False, False], [False, True]])\n\nlabels = rao_blackwell_targets(test_achieved, test_done, test_fatal,\n                               gamma=0.9, gamma_term=0.8, k_max=2)\nassert labels.shape == (2, 2, 4)\nassert torch.allclose(labels.sum(-1), torch.ones(2, 2), atol=1e-6)\nassert torch.all(labels >= -1e-6)\nassert labels[1, 1, -1] == 1.0  # Фатальное действие без достигнутых целей.\n\n# При гибели следующая цель получает (1-γ_term), при успехе — всю массу.\ntest_positive = torch.ones(1, 2, 1)\ntest_end = torch.ones(1, 2, dtype=torch.bool)\ntest_is_fatal = torch.tensor([[True, False]])\ntest_terminal = rao_blackwell_targets(\n    test_positive, test_end, test_is_fatal, gamma=0.9, gamma_term=0.8\n)\nassert torch.allclose(t

In [ ]:
# Независимые среды с разными seed; при reset в середине цикла seed не задаём.
envs = [gym.make(ENV_ID, max_steps=MAX_EPISODE_STEPS) for _ in range(NUM_ENVS)]
initial_obs = [env.reset(seed=SEED + i)[0] for i, env in enumerate(envs)]
positions = [object_position(envs[i]) for i in range(NUM_ENVS)]
packed = [observe_converter(obs, env) for obs, env in zip(initial_obs, envs)]
images = np.stack([pair[0] for pair in packed])
extras = np.stack([pair[1] for pair in packed])

teacher = Teacher().to(DEVICE)
student = Student().to(DEVICE)
teacher_optimizer = torch.optim.RAdam(teacher.parameters(), lr=LEARNING_RATE)
student_optimizer = torch.optim.Adam(student.parameters(), lr=LEARNING_RATE)
# Независимый масштаб градиентов для каждого оптимизатора.
teacher_scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP_ENABLED)
student_scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP_ENABLED)

# Командуем только цели, с которыми агент хотя бы раз сталкивался.
seen_goals = np.zeros(NUM_GOALS, dtype=bool)
for i, extra in enumerate(extras):
    seen_goals |= goal_predicates(envs[i], extra, positions[i]).astype(bool)

commanded_goals = np.array([
    rng.choice(np.flatnonzero(seen_goals)) for _ in range(NUM_ENVS)
], dtype=np.int64)

# Для понятных метрик отдельно отслеживаем достижение команды внутри эпизода.
episode_reached = np.zeros(NUM_ENVS, dtype=bool)
for i in range(NUM_ENVS):
    episode_reached[i] = bool(goal_predicates(envs[i], extras[i], positions[i])[commanded_goals[i]])

training_log = {"teacher_loss": [], "ppo_loss": [], "bc_loss": [],
                "episode_success": [], "eval_updates": [], "teacher_eval": []}
print("Models initialized.")

In [ ]:
from copy import deepcopy

@torch.no_grad()
def evaluate_teacher(teacher, student, episodes=10, every=8, seed=42):
    """CRL-качество будущих целей + точность one-step ранжирования действий.

    Траектории собирает stochastic Student, Teacher лишь предсказывает.
    """
    if episodes < 1 or every < 1:
        raise ValueError("episodes и every должны быть положительными")

    goal_pool = np.flatnonzero(seen_goals)
    if len(goal_pool) == 0:
        raise ValueError("seen_goals пока пусто")
    teacher_mode, student_mode = teacher.training, student.training
    rng_eval = np.random.default_rng(seed)

    losses, kls = [], []
    weighted_loss, sum_weights = 0., 0.
    event_anchors = 0
    correct, compared, pairwise_sum, pairwise_n = 0, 0, 0., 0
    env = None

    try:
        teacher.eval()
        student.eval()
        env = gym.make(ENV_ID, max_steps=MAX_EPISODE_STEPS)
        for episode in range(episodes):
            obs, _ = env.reset(seed=seed + episode)
            image, extra = observe_converter(obs, env)
            positions = object_position(env)   # Не пересчитывать после pickup!
            commanded_goal = int(rng_eval.choice(goal_pool))

            achieved_seq, events_seq, predictions = [], [], []
            fatal_end = False

            for t in range(MAX_EPISODE_STEPS):
                before = goal_predicates(env, extra, positions).astype(bool)
                image_t = torch.as_tensor(image[None], device=DEVICE)
                extra_t = torch.as_tensor(extra[None], device=DEVICE)
                goal_t = torch.tensor([commanded_goal], device=DEVICE)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                    logits, _ = student(image_t, extra_t, goal_t)
                action_probs = F.softmax(logits[0].float(), dim=-1).double().cpu().numpy()
                action_probs /= action_probs.sum()
                action = int(rng_eval.choice(NUM_ACTIONS, p=action_probs))

                if t % every == 0:
                    with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                        scores = teacher(image_t, extra_t)[0].float().cpu().numpy()  # [A,G]
                    predictions.append((t, scores[action].copy()))

                    # One-step oracle: что произойдёт с каждой целью за 1 шаг?
                    outcomes = np.zeros((NUM_ACTIONS, NUM_GOALS), dtype=bool)
                    for a in range(NUM_ACTIONS):
                        clone = deepcopy(env)
                        try:
                            next_obs, _, _, _, _ = clone.step(a)
                            _, next_extra = observe_converter(next_obs, clone)
                            outcomes[a] = goal_predicates(clone, next_extra, positions)
                        finally:
                            clone.close()

                    gain = outcomes & ~before[None, :]
                    for g in np.flatnonzero(gain.any(axis=0) & ~gain.all(axis=0)):
                        good = gain[:, g]
                        correct += int(good[scores[:, g].argmax()])
                        compared += 1
                        diffs = scores[good, g][:, None] - scores[~good, g][None, :]
                        pairwise_sum += float((diffs > 0).sum() + 0.5 * (diffs == 0).sum())
                        pairwise_n += diffs.size

                obs, reward, terminated, truncated, _ = env.step(action)
                image, extra = observe_converter(obs, env)
                achieved = goal_predicates(env, extra, positions).astype(bool)
                achieved_seq.append(achieved)
                events_seq.append(achieved & ~before)  # Новое достижение: 0 -> 1

                if achieved[commanded_goal] and not (terminated or truncated):
                    commanded_goal = int(rng_eval.choice(goal_pool))
                if terminated or truncated:
                    fatal_end = bool(terminated and reward <= 0)
                    break

            achieved_seq = np.asarray(achieved_seq, dtype=np.float64)
            events_seq = np.asarray(events_seq, dtype=bool)
            T = len(achieved_seq)

            for t, score in predictions:
                future = achieved_seq[t:min(T, t + K_MAX)]
                weights = GAMMA ** np.arange(len(future))
                labels = future / np.maximum(1, future.sum(axis=1, keepdims=True))
                target = (weights[:, None] * labels).sum(axis=0) / weights.sum()
                if fatal_end and T - t <= K_MAX:
                    target *= 1.0 - GAMMA_TERM ** (T - t)
                target = np.append(target, max(0., 1. - target.sum()))

                logits = np.append(score.astype(np.float64), 0.)  # logit(∅)=0
                # log-softmax без underflow вероятностей и искусственного потолка CE.
                shifted = logits - logits.max()
                log_probs = shifted - np.log(np.exp(shifted).sum())
                ce = -np.sum(target * log_probs)
                mask = target > 0
                kl = np.sum(target[mask] * (np.log(target[mask]) - log_probs[mask]))
                losses.append(float(ce))
                kls.append(float(kl))

                # Доп. метрика пользователя: чем дальше ближайшее новое достижение,
                # тем меньше вклад ошибки (вес 1 / число шагов до него).
                future_events = np.flatnonzero(events_seq[t:min(T, t + K_MAX)].any(axis=1))
                if len(future_events):
                    w = 1. / (int(future_events[0]) + 1)
                    weighted_loss += w * ce
                    sum_weights += w
                    event_anchors += 1
    finally:
        try:
            if env is not None:
                env.close()
        finally:
            teacher.train(teacher_mode)
            student.train(student_mode)

    return {
        "ce": float(np.mean(losses)),
        "kl": float(np.mean(kls)),
        "distance_weighted_ce": weighted_loss / sum_weights if sum_weights else np.nan,
        "event_coverage": event_anchors / len(losses),
        "one_step_accuracy": correct / compared if compared else np.nan,
        "one_step_pairwise": pairwise_sum / pairwise_n if pairwise_n else np.nan,
        "ranking_cases": compared,
        "prediction_cases": len(losses),
    }


In [ ]:
@torch.no_grad()
def evaluate(agent, episodes_per_goal=3, seed=10_000):
    """Считает greedy goal success для Teacher или Student.

    Задача:
        Отделить качество достижения цели от PPO training reward.
        Для каждого goal_id запускает несколько независимых эпизодов.

    Аргументы:
        agent: экземпляр Teacher (действие по f) или Student (по π);
        episodes_per_goal: число повторений каждой цели;
        seed: фиксирует идентичные стартовые условия для сравнения.

    Возвращает:
        success_rates [NUM_GOALS] — доля эпизодов достижения каждой цели.

    Примечание:
        В обучении цели ставятся лишь из набора seen_goals; здесь мы
        намеренно проверяем ВСЁ известное пространство G.
    """
    agent.eval()
    success_rates = np.zeros(NUM_GOALS, dtype=np.float32)
    env = gym.make(ENV_ID, max_steps=MAX_EPISODE_STEPS)


    for goal_id in range(NUM_GOALS):
        successes = []

        for episode in range(episodes_per_goal):
            # Повторяем выбор стартового состояния, если цель выполнена сразу.
            for attempt in range(30):
                observation, _ = env.reset(seed=seed + 1000*goal_id + 50*episode + attempt)
                image, extra = observe_converter(observation, env)
                position = object_position(env)
                if goal_predicates(env, extra, position)[goal_id] == 0:
                    break

            reached = bool(goal_predicates(env, extra, position)[goal_id])

            for _ in range(MAX_EPISODE_STEPS):
                image_t = torch.as_tensor(image[None], device=DEVICE)
                extra_t = torch.as_tensor(extra[None], device=DEVICE)
                goal_t = torch.tensor([goal_id], device=DEVICE)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                    logits, _ = agent(image_t, extra_t, goal_t)
                action = logits.argmax(dim=-1).item()

                observation, _, terminated, truncated, _ = env.step(action)
                image, extra = observe_converter(observation, env)
                reached |= bool(goal_predicates(env, extra, position)[goal_id])

                if terminated or truncated or reached:
                    break

            successes.append(float(reached))

        success_rates[goal_id] = np.mean(successes)

    env.close()
    agent.train()
    return success_rates

"""
student_success = evaluate(student, episodes_per_goal=3)
teacher_success = evaluate(teacher, episodes_per_goal=3)

print(f"Student greedy mean success: {student_success.mean():.3f}")
print(f"Teacher greedy mean success: {teacher_success.mean():.3f}")

for name, st_rate, te_rate in zip(GOAL_NAMES, student_success, teacher_success):
    print(f"{name:23s}  student={st_rate:.2f}  teacher={te_rate:.2f}")
"""

In [ ]:
#goal_counts = np.zeros(NUM_GOALS, dtype=np.int64)
for update in trange(NUM_UPDATES, desc="Dual-CRL-PPO"):
    # ================================================================
    # A. COLLECT ROLLOUT — Student взаимодействует с MiniGrid.
    # ================================================================
    trajectory = {key: [] for key in [
        "image", "extra", "goal", "action", "log_prob", "value",
        "reward", "next_value", "done", "terminated", "fatal", "achieved",
        "goal_reached", "goal_done"
    ]}
    successes_this_rollout = []

    for t in range(ROLLOUT_LENGTH):
        image_tensor = torch.as_tensor(images, device=DEVICE)
        extra_tensor = torch.as_tensor(extras, device=DEVICE)
        goals_tensor = torch.as_tensor(commanded_goals, dtype=torch.long, device=DEVICE)
        step_goals = commanded_goals.copy()

        # Student сам стохастически выбирает действия из π_old(a|s,g).
        with torch.no_grad():
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                logits, values = student(image_tensor, extra_tensor, goals_tensor)
            values = values.float()
            distribution = Categorical(logits=logits.float())
            actions_tensor = distribution.sample()
            old_log_probs = distribution.log_prob(actions_tensor)

        actions_np = actions_tensor.cpu().numpy()
        next_images, next_extras, next_achieved = [], [], []
        rewards, dones, terminations, fatalities = [], [], [], []
        goal_dones, goal_reacheds = [], []

        for i, env in enumerate(envs):
            observation, original_reward, terminated, truncated, _ = env.step(int(actions_np[i]))
            next_image, next_extra = observe_converter(observation, env)
            achieved = goal_predicates(env, next_extra, positions[i])
            goal_reached = achieved[commanded_goals[i]]



            # r(s_{t+1},g) = 1, когда достигнута ИМЕННО командованная цель.
            # Параллельно сохраняем ВСЕ достижения для обучения Teacher.
            rewards.append(float(goal_reached))
            next_images.append(next_image)
            next_extras.append(next_extra)
            next_achieved.append(achieved.astype(bool))
            goal_reacheds.append(bool(goal_reached))
            #goal_counts += next_achieved.astype(bool).sum(axis=0)

            # Success-завершение MiniGrid имеет original_reward>0.
            # Реальная смерть в LavaCrossing имеет original_reward=0.
            done = bool(terminated or truncated)
            fatal = bool(terminated and original_reward <= 0)
            if goal_reached and not done:
                commanded_goals[i] = rng.choice(np.flatnonzero(seen_goals))
            dones.append(done)
            goal_dones.append(bool(goal_reached or done))
            terminations.append(bool(terminated))
            fatalities.append(fatal)

            # После терминального перехода будет reset, но старые
            # next_* должны остаться в rollout (для CRL и GAE).
            if goal_reached or done:
                successes_this_rollout.append(float(goal_reached))

        next_images = np.stack(next_images)
        next_extras = np.stack(next_extras)
        next_achieved = np.stack(next_achieved)

        # V(next state, СТАРАЯ цель) нужно для TD-ошибки PPO.
        with torch.no_grad():
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                _, next_values = student(
                    torch.as_tensor(next_images, device=DEVICE),
                    torch.as_tensor(next_extras, device=DEVICE),
                    goals_tensor,
                )
            next_values = next_values.float()

        trajectory["image"].append(images.copy())
        trajectory["extra"].append(extras.copy())
        trajectory["goal"].append(step_goals)
        trajectory["action"].append(actions_np.copy())
        trajectory["log_prob"].append(old_log_probs.cpu().numpy())
        trajectory["value"].append(values.cpu().numpy())
        trajectory["reward"].append(np.array(rewards, dtype=np.float32))
        trajectory["next_value"].append(next_values.cpu().numpy())
        trajectory["done"].append(np.array(dones, dtype=bool))
        trajectory["terminated"].append(np.array(terminations, dtype=bool))
        trajectory["fatal"].append(np.array(fatalities, dtype=bool))
        trajectory["achieved"].append(next_achieved)
        trajectory["goal_reached"].append(goal_reacheds.copy())
        trajectory["goal_done"].append(goal_dones.copy())

        # Все вновь достигнутые цели становятся допустимыми командами.
        seen_goals |= next_achieved.astype(bool).any(axis=0)

        # Для следующего шага используем reset ТОЛЬКО завершившихся сред.
        images, extras = next_images.copy(), next_extras.copy()
        for i, done in enumerate(dones):
            if done:
                obs, _ = envs[i].reset()
                positions[i] = object_position(envs[i])
                images[i], extras[i] = observe_converter(obs, envs[i])
                seen_goals |= goal_predicates(envs[i], extras[i], positions[i]).astype(bool)

                # Команда неизменна в пределах одного эпизода.
                commanded_goals[i] = rng.choice(np.flatnonzero(seen_goals))
                episode_reached[i] = bool(goal_predicates(envs[i], extras[i], positions[i])[commanded_goals[i]])

    # Превращаем список T массивов размера N в тензор [T,N,...].
    batch = {}
    for key, items in trajectory.items():
        array = np.stack(items)
        dtype = torch.long if key in ("goal", "action") else None
        batch[key] = torch.as_tensor(array, device=DEVICE, dtype=dtype)

    T, N = batch["action"].shape
    B = T * N
    flat_image = batch["image"].reshape(B, 7, 7, 3)
    flat_extra = batch["extra"].reshape(B, 7)
    flat_goals = batch["goal"].reshape(B)
    flat_actions = batch["action"].reshape(B)

    # ================================================================
    # B. TEACHER — обучение на будущем всех достигнутых целей.
    # ================================================================
    targets = rao_blackwell_targets(
        batch["achieved"], batch["done"], batch["fatal"],
        gamma=GAMMA, gamma_term=GAMMA_TERM, k_max=K_MAX,
    ).reshape(B, NUM_GOALS + 1)

    teacher_losses = []
    for _ in range(TEACHER_EPOCHS):
        for indices in torch.randperm(B, device=DEVICE).split(MINIBATCH_SIZE):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss_teacher = teacher.loss(
                    flat_image[indices], flat_extra[indices],
                    flat_actions[indices], targets[indices],
                )
            teacher_optimizer.zero_grad()
            teacher_scaler.scale(loss_teacher).backward()
            teacher_scaler.unscale_(teacher_optimizer)
            nn.utils.clip_grad_norm_(teacher.parameters(), MAX_GRAD_NORM)
            teacher_scaler.step(teacher_optimizer)
            teacher_scaler.update()
            teacher_losses.append(loss_teacher.item())

    # ================================================================
    # C. STUDENT — GAE -> clipped PPO + beta * behavioral cloning.
    # ================================================================
    advantages = torch.zeros(T, N, device=DEVICE)
    running_gae = torch.zeros(N, device=DEVICE)

    for t in reversed(range(T)):
        # При terminated не делаем bootstrap (эпизод действительно завершён).
        # При truncated bootstrap разрешён, но рекурсия по reset обрывается.
        bootstrap = (~batch["terminated"][t] & ~batch["goal_reached"][t]).float()
        continuation = (~batch["goal_done"][t]).float()
        delta = (batch["reward"][t]
                 + GAMMA * bootstrap * batch["next_value"][t]
                 - batch["value"][t])
        running_gae = delta + GAMMA * GAE_LAMBDA * continuation * running_gae
        advantages[t] = running_gae

    returns = (advantages + batch["value"]).reshape(B).detach()
    advantages = advantages.reshape(B)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    old_log_probs = batch["log_prob"].reshape(B)

    # β убывает линейно до 0: в конце Student полагается на свой PPO.
    beta = BC_START * (1.0 - update / max(1, NUM_UPDATES - 1))
    ppo_losses, bc_losses, agreements = [], [], []

    for _ in range(PPO_EPOCHS):
        for indices in torch.randperm(B, device=DEVICE).split(MINIBATCH_SIZE):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                new_logits, new_values = student(
                    flat_image[indices], flat_extra[indices], flat_goals[indices]
                )
            # Вероятности, PPO ratio, value loss и BC остаются в float32.
            new_logits, new_values = new_logits.float(), new_values.float()
            distribution = Categorical(logits=new_logits)
            new_log_probs = distribution.log_prob(flat_actions[indices])

            # Стандартная clipped PPO objective.
            ratio = (new_log_probs - old_log_probs[indices]).exp()
            surrogate_1 = ratio * advantages[indices]
            surrogate_2 = torch.clamp(ratio, 1-PPO_CLIP, 1+PPO_CLIP) * advantages[indices]
            policy_loss = -torch.minimum(surrogate_1, surrogate_2).mean()
            value_loss = 0.5 * (new_values - returns[indices]).square().mean()
            entropy = distribution.entropy().mean()
            ppo_loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            # BC: учитель даёт argmax-действие для той же (s,g).
            # Здесь нет градиентов через Teacher.
            with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                teacher_actions = teacher.greedy_action(
                    flat_image[indices], flat_extra[indices], flat_goals[indices]
                )
                agreement = (new_logits.argmax(dim=-1) == teacher_actions).float().mean()
            bc_loss = F.cross_entropy(new_logits, teacher_actions)
            total_loss = ppo_loss + beta * bc_loss   # Формула (9).

            student_optimizer.zero_grad()
            student_scaler.scale(total_loss).backward()
            student_scaler.unscale_(student_optimizer)
            nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
            student_scaler.step(student_optimizer)
            student_scaler.update()
            ppo_losses.append(ppo_loss.item())
            bc_losses.append(bc_loss.item())
            agreements.append(agreement.item())

    training_log["teacher_loss"].append(float(np.mean(teacher_losses)))
    training_log["ppo_loss"].append(float(np.mean(ppo_losses)))
    training_log["bc_loss"].append(float(np.mean(bc_losses)))
    training_log["episode_success"].append(
        float(np.mean(successes_this_rollout)) if successes_this_rollout else np.nan
    )

    if update % 25 == 0 or update == NUM_UPDATES - 1:
        print(f"update {update+1}/{NUM_UPDATES}  "
              f"teacher={np.mean(teacher_losses):.3f}  "
              f"PPO={np.mean(ppo_losses):.3f}  "
              f"BC={np.mean(bc_losses):.3f}  "
              f"beta={beta:.3f}  "
              f"BC agreement: {np.mean(agreements):.3f} "
              f"known_goals={seen_goals.sum()}/{NUM_GOALS}")
    if (update+1) % 100 == 0 or update == NUM_UPDATES - 1:
        teacher_metrics = evaluate_teacher(teacher, student)
        student_success = evaluate(student, episodes_per_goal=10)
        training_log["eval_updates"].append(update + 1)
        training_log["teacher_eval"].append(teacher_metrics)

        print(f"\nUpdate {update + 1}")
        for name, s_success in zip(GOAL_NAMES, student_success):
            print(f"{name:23} S={s_success:.2f}")
        for metric, value in teacher_metrics.items():
            print(f"Teacher {metric:23} {value:.4g}")

for env in envs:
    env.close()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

axes[0].plot(training_log["teacher_loss"], label="Teacher CRL CE")
axes[0].set_title("Teacher: soft-label CE")
axes[0].set_xlabel("Update")
axes[0].legend()

axes[1].plot(training_log["ppo_loss"], label="Student PPO")
axes[1].plot(training_log["bc_loss"], label="Student BC")
axes[1].set_title("Student losses")
axes[1].set_xlabel("Update")
axes[1].legend()

positions = np.arange(NUM_GOALS)
axes[2].bar(positions, student_success, .6, label="Student")
axes[2].set_xticks(positions, GOAL_NAMES, rotation=90)
axes[2].set_ylim(0, 1)
axes[2].set_title("Student: greedy success per goal")
axes[2].legend()

# CRL-оценка Teacher возвращает метрики, а не success по отдельным целям.
eval_updates = training_log["eval_updates"]
teacher_evals = training_log["teacher_eval"]
for metric in ("ce", "kl", "distance_weighted_ce"):
    axes[3].plot(eval_updates, [m[metric] for m in teacher_evals], marker="o", label=metric)
axes[3].set_title("Teacher: held-out CRL quality")
axes[3].set_xlabel("Update")
axes[3].legend()

for metric in ("one_step_accuracy", "one_step_pairwise", "event_coverage"):
    axes[4].plot(eval_updates, [m[metric] for m in teacher_evals], marker="o", label=metric)
axes[4].set_ylim(0, 1)
axes[4].set_title("Teacher: one-step ranking / event coverage")
axes[4].set_xlabel("Update")
axes[4].legend()

for metric in ("ranking_cases", "prediction_cases"):
    axes[5].plot(eval_updates, [m[metric] for m in teacher_evals], marker="o", label=metric)
axes[5].set_title("Teacher: evaluation sample counts")
axes[5].set_xlabel("Update")
axes[5].legend()

plt.tight_layout()
plt.show()

# Веса + список целей: без него сопоставление id <-> смысл цели потеряется.
checkpoint_path = "dual_crl_ppo_minigrid.pt"
torch.save({
    "teacher": teacher.state_dict(),
    "student": student.state_dict(),
    "goal_names": GOAL_NAMES,
    "config": {
        "env_id": ENV_ID, "grid_size": GRID_SIZE,
        "gamma": GAMMA, "gamma_term": GAMMA_TERM, "k_max": K_MAX,
        "hidden": HIDDEN, "embedding_dim": EMBED_DIM,
    },
}, checkpoint_path)
print("Saved:", checkpoint_path)